# 🧠 Session 2: Embeddings, evaluation and an app (solution)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pearlin-in/recommendationsystem---GDGHWU/blob/main/notebooks/session2_solution.ipynb)

**GDG on Campus · Build Your Own Recommendation System**

This is the **solution notebook**: every ✏️ TODO is already filled in. Use it to catch up if you fall behind, or to compare with your own answers.
The version with blanks to fill in is [`session2_starter.ipynb`](https://colab.research.google.com/github/pearlin-in/recommendationsystem---GDGHWU/blob/main/notebooks/session2_starter.ipynb).

| Part | What you will do | ⏱ |
|---|---|---|
| 0 | Setup and fast-forward through Session 1 | 5 min |
| 1 | Sentence embeddings for every movie | 15 min |
| 2 | TF-IDF vs embeddings, side by side | 10 min |
| 3 | Evaluate: which method is better? | 15 min |
| 4 | ✨ Gemini explains the recommendations | 10 min |
| 5 | Ship it: a shareable Gradio app | 15 min |
| 6 | Your turn: songs 🎵 or books 📚 | 20 min |
| 7 | Optional: visualise the embedding space | optional |

**Deliverable:** a shareable recommender app you can publish to GitHub and put on your CV.

---
## Part 0 · Setup

**Runtime:** the default CPU is fine. If **Runtime > Change runtime type > T4 GPU** is available, it makes Part 1 faster, but you don't need it.

We need two libraries that Colab doesn't include, plus Google's Gemini SDK for Part 4 (usually preinstalled, listed here to be safe).
If Colab asks you to **restart the session** after installing, click **Runtime > Restart session**, then run the cells below this one.

In [ ]:
!pip install -q sentence-transformers gradio google-genai

In [ ]:
import os, io, json, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 100)
REPO_RAW = "https://raw.githubusercontent.com/pearlin-in/recommendationsystem---GDGHWU/main/"

try:
    import google.colab  # only exists inside Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def check(test, hint):
    """Tiny auto-grader used by the 🧪 Check cells."""
    try:
        assert test()
        print("✅ Nice, that works!")
    except Exception:
        print("❌ Not quite yet. Hint:", hint)

### Optional: save your work to Google Drive

Computing embeddings takes a few minutes on CPU. If the Colab runtime disconnects, everything in memory is lost.
Set `USE_DRIVE = True` to save the embeddings in your Google Drive (in a `recsys-workshop` folder), so you don't recompute them after a disconnect.

In [ ]:
USE_DRIVE = False  # ✏️ set to True to save results in your Google Drive

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = "/content/drive/MyDrive/recsys-workshop"
else:
    CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)
print("Embeddings will be saved in:", CACHE_DIR)

In [ ]:
#@title ⏩ Fast-forward: Session 1 in one cell (just run it) { display-mode: "form" }
from datasets import load_dataset


def names(cell):
    try:
        return [d["name"] for d in json.loads(cell)]
    except Exception:
        return []


try:
    raw = load_dataset("AiresPucrs/tmdb-5000-movies", split="train").to_pandas()
except Exception as e:
    print("Hugging Face download failed, using the backup copy:", e)
    raw = pd.read_parquet(REPO_RAW + "data/tmdb_5000_movies.parquet")

movies = raw.copy()
movies["overview"] = movies["overview"].fillna("")
movies["genre_list"] = movies["genres"].apply(names)
movies["keyword_list"] = movies["keywords"].apply(names)
movies["year"] = pd.to_datetime(movies["release_date"], errors="coerce").dt.year.fillna(0).astype(int)
movies["soup"] = (
    movies["overview"]
    + " Genres: " + movies["genre_list"].apply(", ".join)
    + ". Keywords: " + movies["keyword_list"].apply(", ".join)
)
movies = movies.sort_values("vote_count", ascending=False, kind="stable").drop_duplicates("title")
movies = movies[movies["soup"].str.len() > 30].reset_index(drop=True)

tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=20000)
X = tfidf.fit_transform(movies["soup"])


def find_index(query, df, title_col="title"):
    """Row position of the item whose title matches `query` (case-insensitive)."""
    titles = df[title_col].str.lower()
    matches = np.flatnonzero(titles == query.lower())
    if len(matches) == 0:
        close = df.loc[titles.str.contains(query.lower(), regex=False), title_col].head(5).tolist()
        raise ValueError(f"'{query}' not found. Did you mean: {close}")
    return int(matches[0])


def recommend(query, vectors, df, title_col="title", k=5):
    """Return the k items most similar to `query` (works for sparse or dense vectors)."""
    i = find_index(query, df, title_col)
    sims = cosine_similarity(vectors[i:i+1], vectors).ravel()
    sims[i] = -np.inf
    top = np.argsort(-sims)[:k]
    out = df.iloc[top].copy()
    out["similarity"] = sims[top].round(3)
    return out


print(f"{len(movies)} movies, TF-IDF matrix {X.shape}")
recommend("The Dark Knight", X, movies)[["title", "year", "similarity"]]

---
## Part 1 · Sentence embeddings for every movie

A **sentence embedding model** is a neural network that turns a piece of text into a fixed-size vector, so that texts with **similar meaning point in similar directions**, even if they share no words.

We use [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2): small (about 90 MB), CPU-friendly, and it outputs **384 numbers** per text.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # downloads about 90 MB the first time

Encoding all movies is one line: `model.encode(movies["soup"].tolist(), batch_size=64, normalize_embeddings=True)`.
On a free CPU runtime it takes a few minutes, so the helper below **reuses a saved copy** when it can: first your `CACHE_DIR` (your Google Drive, if you turned that on), then the precomputed file in this repo's `embeddings/` folder.
Only if neither exists does it compute the embeddings, and then it saves them for next time.

> `normalize_embeddings=True` scales every vector to length 1, so cosine similarity becomes a simple dot product.

In [ ]:
#@title 🔧 Helper: load_or_compute_embeddings() { display-mode: "form" }
def load_or_compute_embeddings(name, texts, model, batch_size=64):
    """Return normalized embeddings for `texts`, reusing a saved copy when one matches."""
    cache_path = os.path.join(CACHE_DIR, f"{name}.npy")
    sources = [cache_path, f"../embeddings/{name}.npy", REPO_RAW + f"embeddings/{name}.npy"]
    probe = [0, len(texts) // 2, len(texts) - 1]  # rows used to spot-check a saved file
    probe_emb = model.encode([texts[i] for i in probe], normalize_embeddings=True)
    for src in sources:
        try:
            if src.startswith("http"):
                with urllib.request.urlopen(src, timeout=60) as r:
                    emb = np.load(io.BytesIO(r.read()))
            else:
                emb = np.load(src)
            emb = emb.astype("float32")
            if len(emb) == len(texts) and (emb[probe] * probe_emb).sum(axis=1).min() > 0.98:
                print(f"⏩ Loaded {name} from {src}")
                break
        except Exception:
            pass
    else:
        print(f"Computing {name} for {len(texts):,} texts (a few minutes on CPU, much faster on GPU)...")
        emb = model.encode(texts, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)
    if src != cache_path:
        np.save(cache_path, emb.astype("float16"))  # float16 halves the file size, with no visible loss
    return emb

In [ ]:
emb = load_or_compute_embeddings("movies_minilm", movies["soup"].tolist(), model)
print(emb.shape)  # (number of movies, 384)

### ✏️ Exercise 1: recommend with embeddings

`recommend()` from Session 1 works unchanged: it accepts any vectors. Use it with `emb` instead of `X`.

In [ ]:
recs_emb = recommend("The Dark Knight", emb, movies)
recs_emb[["title", "year", "similarity"]]

In [ ]:
# 🧪 Check
check(lambda: emb.shape == (len(movies), 384) and "The Dark Knight Rises" in recs_emb["title"].values,
      'recommend("The Dark Knight", emb, movies)')

---
## Part 2 · TF-IDF vs embeddings, side by side

**TF-IDF matches words. Embeddings match meaning.** Let's put the two result lists next to each other.

In [ ]:
def compare(title, methods, df=None, title_col="title", k=5):
    """Top-k titles of several methods side by side. `methods` is {"name": vectors}."""
    df = movies if df is None else df
    return pd.DataFrame({name: recommend(title, vecs, df, title_col, k)[title_col].values
                         for name, vecs in methods.items()})


for title in ["Titanic", "The Notebook"]:
    print(f"\nBecause you liked {title}:")
    display(compare(title, {"TF-IDF (words)": X, "embeddings (meaning)": emb}))

### ✏️ Exercise 2: find a disagreement

Try a movie whose overview uses unusual vocabulary. Embeddings should find thematically related films that share almost no words.
Ideas: *Finding Nemo*, *Up*, *The Social Network*, *Gravity*, or your own favourites. Which list would you rather get?

In [ ]:
compare("Finding Nemo", {"TF-IDF (words)": X, "embeddings (meaning)": emb})

---
## Part 3 · Evaluate: which method is better?

There are no labels in unsupervised learning, so how can we **measure** quality?
We write down pairs that any good recommender should connect, mostly sequels plus a few harder "same vibe" pairs.
For each pair `(A, B)` we ask: is B in A's top 5? The **hit rate @ 5** is the share of pairs where it is.

In [ ]:
pairs = [
    ("The Dark Knight", "The Dark Knight Rises"), ("Toy Story", "Toy Story 2"), ("The Matrix", "The Matrix Reloaded"),
    ("Shrek", "Shrek 2"), ("Iron Man", "Iron Man 2"), ("Spider-Man", "Spider-Man 2"), ("Alien", "Aliens"),
    ("Harry Potter and the Philosopher's Stone", "Harry Potter and the Chamber of Secrets"),
    ("The Lord of the Rings: The Fellowship of the Ring", "The Lord of the Rings: The Two Towers"),
    ("Pirates of the Caribbean: The Curse of the Black Pearl", "Pirates of the Caribbean: Dead Man's Chest"),
    ("The Terminator", "Terminator 2: Judgment Day"), ("Despicable Me", "Despicable Me 2"),
    ("The Hunger Games", "The Hunger Games: Catching Fire"), ("Star Wars", "The Empire Strikes Back"),
    ("Back to the Future", "Back to the Future Part II"), ("Ice Age", "Ice Age: The Meltdown"),
    ("Kung Fu Panda", "Kung Fu Panda 2"), ("How to Train Your Dragon", "How to Train Your Dragon 2"),
    ("The Godfather", "The Godfather: Part II"), ("Jurassic Park", "The Lost World: Jurassic Park"),
    ("Die Hard", "Die Hard 2"), ("Men in Black", "Men in Black II"), ("The Bourne Identity", "The Bourne Supremacy"),
    ("Cars", "Cars 2"), ("Mad Max: Fury Road", "Mad Max"),
    # harder "same vibe" pairs with no shared franchise name
    ("Frozen", "Tangled"), ("Inception", "Interstellar"), ("The Notebook", "A Walk to Remember"),
]
print(len(pairs), "pairs")

### ✏️ Exercise 3: hit rate @ k

In [ ]:
def hit_rate_at_k(vectors, k=5):
    """Share of pairs (a, b) where b is in a's top k."""
    hits = []
    for a, b in pairs:
        top_titles = recommend(a, vectors, movies, k=k)["title"].values
        hits.append(b in top_titles)
    return np.mean(hits)

In [ ]:
results = pd.Series({"TF-IDF": hit_rate_at_k(X), "embeddings": hit_rate_at_k(emb)}, name="hit rate @5").round(2)
results

In [ ]:
# 🧪 Check
check(lambda: 0.5 < results["TF-IDF"] < 1, "hits.append(b in top_titles)")

**Discuss:**
- Which method wins? Sequels share names, which favours TF-IDF. The "same vibe" pairs are where meaning should help.
- Our 28 pairs are a tiny, hand-made test. In industry, the real test is an **A/B test** with real users: do they click, watch and come back?

---
## Part 4 · ✨ Gemini explains the recommendations

Good products tell you **why** something was recommended. We'll ask Google's **Gemini** to write one line per recommendation.
The trick: we give Gemini the plot summaries our recommender found and tell it to use **only** those.
Our recommender *retrieves*, Gemini *generates*. This pattern is called **Retrieval-Augmented Generation (RAG)**.

**Two free ways to use Gemini (pick one):**
1. **Nothing to set up (Colab only):** the helper below falls back to Colab's built-in Gemini access (`google.colab.ai`), which needs no API key.
2. **Your own API key (works anywhere):** get a free key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey).
   In Colab, click the 🔑 **Secrets** icon in the left sidebar, add a secret named `GOOGLE_API_KEY` with your key as the value, and switch on **Notebook access**.
   Never paste a key directly into notebook code.

No Gemini at all? That's fine: everything else in this notebook still works.

In [ ]:
#@title 🔧 Helper: ask_gemini() { display-mode: "form" }
GEMINI_MODEL = "gemini-flash-latest"  # an alias that always points to Google's latest Flash model


def get_gemini_key():
    """Read the key from Colab Secrets, or from an environment variable outside Colab."""
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get("GOOGLE_API_KEY")
        except Exception:
            pass
    return os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")


GEMINI_KEY = get_gemini_key()
_client = None


def ask_gemini(prompt):
    """Send a prompt to Gemini and return its text answer, or None if Gemini isn't available."""
    global _client
    try:
        if GEMINI_KEY:
            from google import genai
            if _client is None:
                _client = genai.Client(api_key=GEMINI_KEY)
            return _client.models.generate_content(model=GEMINI_MODEL, contents=prompt).text
        if IN_COLAB:
            from google.colab import ai  # Colab's built-in Gemini access, no key needed
            return ai.generate_text(prompt)
    except Exception as e:
        print("Gemini call failed:", str(e)[:300])
    return None


print("Using:", "Gemini API key" if GEMINI_KEY else "Colab built-in Gemini" if IN_COLAB else "no Gemini (that's OK)")
answer = ask_gemini("In one short sentence, what is a recommender system?")
print(answer or "No answer from Gemini. That's OK: the rest of the notebook still works without it.")

### ✏️ Exercise 4: write the prompt

A good prompt says **who** the model is, gives it the **context**, and states exactly **what to write** and **what not to do**.

In [ ]:
def explain(title, recs):
    """Ask Gemini why each recommended movie suits someone who liked `title`."""
    liked = movies.iloc[find_index(title, movies)]
    rec_lines = "\n".join(f"- {m.title} ({m.year}): {m.overview}" for m in recs.itertuples())
    prompt = f"""You are a friendly movie expert.
Someone loved "{liked.title}" ({liked.year}). Plot: {liked.overview}

Our recommender suggested these movies:
{rec_lines}

For each suggested movie, write one markdown bullet: **Title**, then one sentence (max 25 words) on why a fan
of "{liked.title}" would enjoy it. Use ONLY the plot information above and do not invent facts."""
    return ask_gemini(prompt)

In [ ]:
recs = recommend("Interstellar", emb, movies)
display(recs[["title", "year", "similarity"]])
answer = explain("Interstellar", recs)
display(Markdown(answer) if answer else Markdown("*(Gemini isn't set up, so there is no explanation. See the instructions above.)*"))

---
## Part 5 · Ship it: a shareable Gradio app

[Gradio](https://www.gradio.app) turns a Python function into a web app. The dropdown is searchable, so users can type to filter.
`share=True` prints a **public link** you can send to friends. It is temporary and only works while this notebook is running.

### ✏️ Exercise 5: connect the app to your recommender

In [ ]:
import gradio as gr


def app_fn(choice, use_gemini):
    result = recommend(choice, emb, movies)  # swap in X and the right dataframe as needed
    why = (explain(choice, result) or "*Gemini isn't available right now.*") if use_gemini else ""
    return result[["title", "year", "similarity"]], why

In [ ]:
# 🧪 Check
check(lambda: len(app_fn("Toy Story", False)[0]) == 5, "result = recommend(choice, emb, movies)")

In [ ]:
demo = gr.Interface(
    fn=app_fn,
    inputs=[
        gr.Dropdown(choices=sorted(movies["title"].tolist()), label="Pick or type a movie you like"),
        gr.Checkbox(label="✨ Explain with Gemini"),
    ],
    outputs=[gr.Dataframe(label="Top 5 recommendations"), gr.Markdown()],
    title="Movie Recommender",
    description="Type a movie you like and get 5 similar ones.",
)
demo.launch(share=True)  # prints a temporary public link. Stop the cell (■) to shut the app down.

---
## Part 6 · Your turn: songs 🎵 or books 📚

Pick **one** option. The same `recommend()` works for any vectors.

### Option A: songs 🎵 (vectors from numbers, not text)

The Spotify Tracks dataset has no lyrics or descriptions. Each song has **audio features** instead, such as danceability, energy and tempo.
So songs are represented by *numeric audio vectors*. Vectors don't have to come from text, and cosine similarity works the same way.

In [ ]:
URL = "https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset/resolve/main/dataset.csv"
try:
    songs = pd.read_csv(URL, index_col=0)
except Exception as e:
    print("Download failed, using the backup copy:", e)
    songs = pd.read_csv(REPO_RAW + "data/spotify_tracks.csv.gz", index_col=0)

# 1. Clean: one row per track, keep the more popular ones so results feel familiar
songs = songs.drop_duplicates("track_id")
songs = songs.drop_duplicates(["track_name", "artists"])
songs = songs.dropna(subset=["track_name", "artists"])
songs = songs.sort_values("popularity", ascending=False).head(30000).reset_index(drop=True)

# 2. A readable label to search by
songs["label"] = songs["track_name"] + " - " + songs["artists"]

print(songs.shape)
songs[["label", "track_genre", "popularity", "danceability", "energy", "tempo"]].head()

**Scaling matters.** Tempo is around 120 and loudness around −8, while the other features are between 0 and 1.
Without scaling, tempo and loudness would dominate. `StandardScaler` gives every feature mean 0 and standard deviation 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

features = ["danceability", "energy", "loudness", "speechiness",
            "acousticness", "instrumentalness", "liveness", "valence", "tempo"]
S = StandardScaler().fit_transform(songs[features])

recommend(songs["label"].iloc[0], S, songs, title_col="label")[["label", "track_genre", "similarity"]]

**Try it:** add the genre as an extra feature (one-hot encode `track_genre` with `pd.get_dummies` and concatenate) and see how the recommendations change.

In [ ]:
genre_onehot = pd.get_dummies(songs["track_genre"]).to_numpy(dtype=float)
S_genre = np.hstack([S, genre_onehot])

compare(songs["label"].iloc[0], {"audio only": S, "audio + genre": S_genre}, df=songs, title_col="label")

> Because the dataset has no lyrics, sentence embeddings are **not** a good fit here. Choosing the right representation for the data is a core ML skill.
> Each query compares 1 song with 30,000 others, which takes milliseconds. We never build the full similarity matrix.

### Option B: books 📚 (description and genres)

Goodbooks-10k has 10,000 popular Goodreads books with descriptions and genres, so we reuse exactly the same pipeline as movies.

In [ ]:
from ast import literal_eval

URL = "https://raw.githubusercontent.com/malcolmosh/goodbooks-10k/master/books_enriched.csv"
convert = {"genres": literal_eval, "authors": literal_eval}  # these columns hold lists stored as text
try:
    books = pd.read_csv(URL, index_col=[0], converters=convert)
except Exception as e:
    print("Download failed, using the backup copy:", e)
    books = pd.read_csv(REPO_RAW + "data/books_enriched.csv.gz", index_col=[0], converters=convert)
print(books.columns.tolist())  # confirm the column names first

books["description"] = books["description"].fillna("")
books["author_str"] = books["authors"].apply(", ".join)
# Genres go before the description: the embedding model only reads the first ~256 words
books["soup"] = (books["title"] + ". By " + books["author_str"] + ". Genres: "
                 + books["genres"].apply(", ".join) + ". " + books["description"])
books = books.sort_values("ratings_count", ascending=False, kind="stable").drop_duplicates("title")
books = books[books["soup"].str.len() > 40].reset_index(drop=True)
print(books.shape)

In [ ]:
Xb = TfidfVectorizer(stop_words="english", max_features=20000).fit_transform(books["soup"])
emb_b = load_or_compute_embeddings("books_minilm", books["soup"].tolist(), model)  # precomputed in the repo

compare("The Hobbit", {"TF-IDF": Xb, "embeddings": emb_b}, df=books)

---
## Part 7 · Optional: visualise the embedding space

Our embeddings have 384 dimensions. **UMAP** squashes them down to 2 so we can look at them, keeping nearby movies nearby.

In [ ]:
!pip install -q umap-learn

In [ ]:
import umap

coords = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42, n_jobs=1).fit_transform(emb)
plt.figure(figsize=(8, 6))
plt.scatter(coords[:, 0], coords[:, 1], s=3, alpha=0.5)
plt.title("Movie embeddings projected to 2D")
plt.show()

---
## ✅ Recap

| You learned | In one line |
|---|---|
| Sentence embeddings | Vectors that capture *meaning*, not just words |
| Comparing methods | Side-by-side lists and hit rate @5 on known pairs |
| RAG with Gemini | Retrieve with your recommender, explain with an LLM, grounded in real data |
| Gradio | From notebook to shareable web app in a few lines |

### 🚀 Publish your work
In Colab, **File > Save a copy in GitHub**, then add the link to your CV or LinkedIn.

### Stretch goals
- **Hybrid score:** combine TF-IDF and embedding similarity (for example `0.4 * tfidf + 0.6 * embedding`).
- **Multi-item queries:** average the vectors of 3 favourite movies and recommend from that.
- **Filters:** restrict results by genre, year or minimum rating.
- **Popularity bias:** discuss why popular items dominate, and try re-ranking by `vote_count`.
- **Collaborative filtering:** use MovieLens ratings and compare the two approaches.
- **Deploy permanently:** host the app on Hugging Face Spaces (free) instead of a temporary Gradio link.
- **Better text models:** try `all-mpnet-base-v2` (better quality, slower).

*Thanks for building with GDG on Campus! 💙*